In [153]:

import math
from collections.abc import Iterable


class Problem[S, A]:
    def __init__(self, initial_state, goal_state) -> None:
        self.initial_state = initial_state
        self.goal_state = goal_state

    def get_actions(self, state: S) -> Iterable[S]:
        raise NotImplementedError

    def apply_action(self, state: S, action: A) -> S:
        raise NotImplementedError

    def goal_test(self, state: S) -> bool:
        return self.goal_state == state

    def action_cost(self, state1: S, action: A, state2: S) -> float:
        return 1

    def h(self, state: S) -> float:
        return 0.0

In [154]:
class GraphAStarProblem(Problem[str, str]):
  def __init__(
      self, initial: str, goal: str, graph: dict[str, dict[str, float]]
  ) -> None:
    super().__init__(initial, goal)
    self.graph: dict[str, dict[str, float]] = graph

  def get_actions(self, state: str) -> list[str]:
    return list(self.graph[state].keys())

  def apply_action(self, state: str, action: str) -> str:
    return action

  def action_cost(self, state1: str, action: str, state2: str) -> float:
    return self.graph[state1][state2]

  def h(self, state: str) -> float:
    return float(straight_line_distance[state])

In [155]:
distances_of_cities_in_romania: dict[str, dict[str, int]] = {
    "Arad": {"Zerind": 75, "Sibiu": 140, "Timisoara": 118},
    "Zerind": {"Arad": 75, "Oradea": 71},
    "Oradea": {"Zerind": 71, "Sibiu": 151},
    "Sibiu": {"Arad": 140, "Oradea": 151, "Fagaras": 99, "Rimnicu Vilcea": 80},
    "Timisoara": {"Arad": 118, "Lugoj": 111},
    "Lugoj": {"Timisoara": 111, "Mehadia": 70},
    "Mehadia": {"Lugoj": 70, "Dobreta": 75},
    "Dobreta": {"Mehadia": 75, "Craiova": 120},
    "Craiova": {"Dobreta": 120, "Rimnicu Vilcea": 146, "Pitesti": 138},
    "Rimnicu Vilcea": {"Sibiu": 80, "Craiova": 146, "Pitesti": 97},
    "Fagaras": {"Sibiu": 99, "Bucarest": 211},
    "Pitesti": {"Rimnicu Vilcea": 97, "Craiova": 138, "Bucarest": 101},
    "Bucarest": {"Fagaras": 211, "Pitesti": 101, "Giurgiu": 90, "Urziceni": 85},
    "Giurgiu": {"Bucarest": 90},
    "Urziceni": {"Bucarest": 85, "Hirsova": 98, "Vaslui": 142},
    "Hirsova": {"Urziceni": 98, "Eforie": 86},
    "Eforie": {"Hirsova": 86},
    "Vaslui": {"Urziceni": 142, "Iasi": 92},
    "Iasi": {"Vaslui": 92, "Neamt": 87},
    "Neamt": {"Iasi": 87},
}

straight_line_distance = {
    'Arad': 366,
    'Bucarest': 0,
    'Craiova': 160,
    'Dobreta': 242,
    'Eforie': 161,
    'Fagaras': 178,
    'Giurgiu': 77,
    'Hirsova': 151,
    'Iasi': 226,
    'Lugoj': 244,
    'Mehadia': 241,
    'Neamt': 234,
    'Oradea': 380,
    'Pitesti': 98,
    'Rimnicu Vilcea': 193,
    'Sibiu': 253,
    'Timisoara': 329,
    'Urziceni': 80,
    'Vaslui': 199,
    'Zerind': 374,
}

In [156]:
class Node[S, A]:
    def __init__(
        self,
        state: S,
        parent: "Node[S, A] | None" = None,
        action: A | None = None,
        path_cost: float = 0,
    ) -> None:
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def path(self) -> list[S]:
        path_list: list[Node[S]] = []

        node: Node = self
        while node:
            path_list.append(node.state)
            node: Node = node.parent

        path_list.reverse()

        return path_list

    def child_node(self, problem: GraphAStarProblem, action: A) -> "Node[S]":
        next_state = problem.apply_action(self.state, action)
        step_cost = problem.action_cost(self.state, action, next_state)

        return Node(next_state, self, action, self.path_cost + step_cost)

    def expand(self, problem: GraphAStarProblem) -> "list[Node[S]]":
        return [
            self.child_node(problem, action)
            for action in problem.get_actions(self.state)
        ]

In [157]:
def hill_climbing(
    problem: GraphAStarProblem[str, str],
) -> tuple[Node[str, str], bool]:
  current_node: Node[str, str] = Node(problem.initial_state)

  while True:
    if problem.goal_test(current_node.state):
      return current_node, True

    neighbors: list[Node[str, str]] = current_node.expand(problem)
    if not neighbors:
      break

    best_neighbor: Node[str, str] = min(
        neighbors, key=lambda child: problem.h(child.state)
    )

    if problem.h(best_neighbor.state) >= problem.h(current_node.state):
      break

    current_node = best_neighbor

  return current_node, problem.is_goal(current_node.state)

In [158]:
arad_to_bucarest_problem: GraphAStarProblem[str, str] = GraphAStarProblem(
    "Arad", "Bucarest", distances_of_cities_in_romania
)

In [159]:
(hill_climb_result, reached_goal) = hill_climbing(arad_to_bucarest_problem)
node_path = hill_climb_result.path()

print(" -> ".join(node_path))

Arad -> Sibiu -> Fagaras -> Bucarest


In [160]:
def haversine_distance(
    coord1: tuple[float, float], coord2: tuple[float, float]
) -> float:
  """Calcula la distancia geodésica en kilómetros entre dos puntos (latitud, longitud)

  usando la fórmula del semiverseno (Haversine).
  """
  R = 6371.0  # Radio medio de la Tierra en km

  lat1, lon1 = math.radians(coord1[0]), math.radians(coord1[1])
  lat2, lon2 = math.radians(coord2[0]), math.radians(coord2[1])

  dlat = lat2 - lat1
  dlon = lon2 - lon1

  a = (
      math.sin(dlat / 2) ** 2
      + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
  )
  c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

  return R * c

In [161]:
metro_lines = [
    # 1
    [
        "Observatorio",
        "Tacubaya",
        "Juanacatlán",
        "Chapultepec",
        "Sevilla",
        "Insurgentes",
        "Cuauhtémoc",
        "Balderas",
        "Salto del Agua",
        "Isabel la Católica",
        "Pino Suárez",
        "Merced",
        "Candelaria",
        "San Lázaro",
        "Moctezuma",
        "Balbuena",
        "Boulevard Puerto Aéreo",
        "Gómez Farías",
        "Zaragoza",
        "Pantitlán",
    ],
    # 2
    [
        "Cuatro Caminos",
        "Panteones",
        "Tacuba",
        "Cuitláhuac",
        "Popotla",
        "Colegio Militar",
        "Normal",
        "San Cosme",
        "Revolución",
        "Hidalgo",
        "Bellas Artes",
        "Allende",
        "Zócalo",
        "Pino Suárez",
        "San Antonio Abad",
        "Chabacano",
        "Viaducto",
        "Xola",
        "Villa de Cortés",
        "Nativitas",
        "Portales",
        "Ermita",
        "General Anaya",
        "Taxqueña",
    ],
    # 3
    [
        "Indios Verdes",
        "18 de Marzo",
        "Potrero",
        "La Raza",
        "Tlatelolco",
        "Guerrero",
        "Hidalgo",
        "Juárez",
        "Balderas",
        "Niños Héroes",
        "Hospital General",
        "Centro Médico",
        "Etiopía",
        "Eugenia",
        "División del Norte",
        "Zapata",
        "Coyoacán",
        "Viveros",
        "M.A. de Quevedo",
        "Copilco",
        "Universidad",
    ],
    # 4
    [
        "Martín Carrera",
        "Talismán",
        "Bondojito",
        "Consulado",
        "Morelos",
        "Candelaria",
        "Fray Servando",
        "Jamaica",
        "Santa Anita",
    ],
    # 5
    [
        "Politécnico",
        "Instituto del Petróleo",
        "Autobuses del Norte",
        "La Raza",
        "Misterios",
        "Vallejo",
        "Consulado",
        "Eduardo Molina",
        "Aragón",
        "Oceanía",
        "Terminal Aérea",
        "Hangares",
        "Pantitlán",
    ],
    # 6
    [
        "El Rosario",
        "Tezozómoc",
        "Azcapotzalco",
        "Ferrería",
        "Norte 45",
        "Vallejo",
        "Instituto del Petróleo",
        "Lindavista",
        "18 de Marzo",
        "La Villa-Basílica",
        "Martín Carrera",
    ],
    # 7
    [
        "El Rosario",
        "Aquiles Serdán",
        "Camarones",
        "Refinería",
        "Tacuba",
        "San Joaquín",
        "Polanco",
        "Auditorio",
        "Constituyentes",
        "Tacubaya",
        "San Pedro de los Pinos",
        "San Antonio",
        "Mixcoac",
        "Barranca del Muerto",
    ],
    # 8
    [
        "Garibaldi",
        "Bellas Artes",
        "San Juan de Letrán",
        "Salto del Agua",
        "Doctores",
        "Obrero Mundial",
        "Chabacano",
        "La Viga",
        "Santa Anita",
        "Coyuya",
        "Iztacalco",
        "Apatlaco",
        "Aculco",
        "Escuadrón 201",
        "Atlalilco",
        "Iztapalapa",
        "Cerro de la Estrella",
        "UAM-I",
        "Constitución de 1917",
    ],
    # 9
    [
        "Tacubaya",
        "Patriotismo",
        "Chilpancingo",
        "Centro Médico",
        "Lázaro Cárdenas",
        "Chabacano",
        "Jamaica",
        "Mixiuhca",
        "Velódromo",
        "Ciudad Deportiva",
        "Puebla",
        "Pantitlán",
    ],
    # A
    [
        "Pantitlán",
        "Agrícola Oriental",
        "Canal de San Juan",
        "Tepalcates",
        "Guelatao",
        "Peñón Viejo",
        "Acatitla",
        "Santa Marta",
        "Los Reyes",
        "La Paz",
    ],
    # B
    [
        "Buenavista",
        "Guerrero",
        "Garibaldi",
        "Lagunilla",
        "Tepito",
        "Morelos",
        "San Lázaro",
        "Ricardo Flores Magón",
        "Romero Rubio",
        "Oceanía",
        "Deportivo Oceanía",
        "Bosque de Aragón",
        "Villa de Aragón",
        "Nezahualcóyotl",
        "Impulsora",
        "Río de los Remedios",
        "Múzquiz",
        "Ecatepec",
        "Olímpica",
        "Plaza Aragón",
        "Ciudad Azteca",
    ],
    # 12
    [
        "Mixcoac",
        "Insurgentes Sur",
        "Hospital 20 de Noviembre",
        "Zapata",
        "Parque de los Venados",
        "Eje Central",
        "Ermita",
        "Mexicaltzingo",
        "Atlalilco",
        "Culhuacán",
        "San Andrés Tomatlán",
        "Lomas Estrella",
        "Calle 11",
        "Periférico Oriente",
        "Tezonco",
        "Olivos",
        "Nopalera",
        "Zapotitlán",
        "Tlaltenco",
        "Tláhuac",
    ],
]

metro_graph: dict[str, set[str]] = {}
for line in metro_lines:
  for i in range(len(line)):
    station = line[i]
    if station not in metro_graph:
      metro_graph[station] = set()
    if i > 0:
      metro_graph[station].add(line[i - 1])
    if i < len(line) - 1:
      metro_graph[station].add(line[i + 1])

In [162]:
station_coordinates: dict[str, tuple[float, float]] = {
    "Observatorio": (19.3982, -99.2004),
    "Tacubaya": (19.4032, -99.1871),
    "Juanacatlán": (19.4129, -99.1821),
    "Chapultepec": (19.4206, -99.1764),
    "Sevilla": (19.4219, -99.1706),
    "Insurgentes": (19.4232, -99.1631),
    "Cuauhtémoc": (19.4258, -99.1547),
    "Balderas": (19.4273, -99.1491),
    "Salto del Agua": (19.4269, -99.1422),
    "Isabel la Católica": (19.4266, -99.1376),
    "Pino Suárez": (19.4257, -99.1332),
    "Merced": (19.4255, -99.1247),
    "Candelaria": (19.4288, -99.1194),
    "San Lázaro": (19.4303, -99.1148),
    "Moctezuma": (19.4274, -99.1102),
    "Balbuena": (19.4234, -99.1026),
    "Boulevard Puerto Aéreo": (19.4197, -99.0962),
    "Gómez Farías": (19.4162, -99.0903),
    "Zaragoza": (19.4122, -99.0824),
    "Pantitlán": (19.4152, -99.0722),
    "Cuatro Caminos": (19.4596, -99.2158),
    "Panteones": (19.4587, -99.2031),
    "Tacuba": (19.4595, -99.1887),
    "Cuitláhuac": (19.4574, -99.1818),
    "Popotla": (19.4522, -99.1738),
    "Colegio Militar": (19.4488, -99.1685),
    "Normal": (19.4443, -99.1628),
    "San Cosme": (19.4414, -99.1561),
    "Revolución": (19.4381, -99.1524),
    "Hidalgo": (19.4373, -99.1472),
    "Bellas Artes": (19.4362, -99.1418),
    "Allende": (19.4357, -99.1373),
    "Zócalo": (19.4326, -99.1323),
    "San Antonio Abad": (19.4187, -99.1343),
    "Chabacano": (19.4084, -99.1356),
    "Viaducto": (19.4009, -99.1368),
    "Xola": (19.3952, -99.1378),
    "Villa de Cortés": (19.3878, -99.1388),
    "Nativitas": (19.3795, -99.1398),
    "Portales": (19.3698, -99.1412),
    "Ermita": (19.3619, -99.1429),
    "General Anaya": (19.3533, -99.1449),
    "Taxqueña": (19.3441, -99.1434),
    "Indios Verdes": (19.4954, -99.1195),
    "18 de Marzo": (19.4837, -99.1263),
    "Potrero": (19.4771, -99.1313),
    "La Raza": (19.4702, -99.1368),
    "Tlatelolco": (19.4554, -99.1425),
    "Guerrero": (19.4452, -99.1454),
    "Juárez": (19.4332, -99.1481),
    "Niños Héroes": (19.4192, -99.1502),
    "Hospital General": (19.4132, -99.1534),
    "Centro Médico": (19.4069, -99.1557),
    "Etiopía": (19.3957, -99.1563),
    "Eugenia": (19.3858, -99.1572),
    "División del Norte": (19.3797, -99.1591),
    "Zapata": (19.3709, -99.1587),
    "Coyoacán": (19.3613, -99.1706),
    "Viveros": (19.3537, -99.1762),
    "M.A. de Quevedo": (19.3458, -99.1812),
    "Copilco": (19.3358, -99.1768),
    "Universidad": (19.3244, -99.1739),
    "Martín Carrera": (19.4851, -99.1044),
    "Talismán": (19.4741, -99.1077),
    "Bondojito": (19.4641, -99.1111),
    "Consulado": (19.4578, -99.1136),
    "Morelos": (19.4388, -99.1192),
    "Fray Servando": (19.4217, -99.1205),
    "Jamaica": (19.4088, -99.1223),
    "Santa Anita": (19.4026, -99.1217),
    "Politécnico": (19.5008, -99.1492),
    "Instituto del Petróleo": (19.4902, -99.1471),
    "Autobuses del Norte": (19.4788, -99.1412),
    "Misterios": (19.4631, -99.1306),
    "Vallejo": (19.4638, -99.1501),
    "Eduardo Molina": (19.4512, -99.1037),
    "Aragón": (19.4507, -99.0963),
    "Oceanía": (19.4457, -99.0872),
    "Terminal Aérea": (19.4336, -99.0881),
    "Hangares": (19.4243, -99.0839),
    "El Rosario": (19.5046, -99.2001),
    "Tezozómoc": (19.4952, -99.1962),
    "Azcapotzalco": (19.4909, -99.1861),
    "Ferrería": (19.4903, -99.1732),
    "Norte 45": (19.4839, -99.1648),
    "Lindavista": (19.4871, -99.1345),
    "La Villa-Basílica": (19.4816, -99.1182),
    "Aquiles Serdán": (19.4905, -99.1948),
    "Camarones": (19.4792, -99.1899),
    "Refinería": (19.4697, -99.1903),
    "San Joaquín": (19.4456, -99.1822),
    "Polanco": (19.4328, -99.1912),
    "Auditorio": (19.4255, -99.1921),
    "Constituyentes": (19.4124, -99.1914),
    "San Pedro de los Pinos": (19.3872, -99.1856),
    "San Antonio": (19.3846, -99.1859),
    "Mixcoac": (19.3758, -99.1874),
    "Barranca del Muerto": (19.3606, -99.1895),
    "Garibaldi": (19.4443, -99.1388),
    "San Juan de Letrán": (19.4316, -99.1417),
    "Doctores": (19.4221, -99.1432),
    "Obrero Mundial": (19.4011, -99.1352),
    "La Viga": (19.4048, -99.1264),
    "Coyuya": (19.3986, -99.1136),
    "Iztacalco": (19.3888, -99.1118),
    "Apatlaco": (19.3787, -99.1105),
    "Aculco": (19.3728, -99.1084),
    "Escuadrón 201": (19.3647, -99.1095),
    "Atlalilco": (19.3562, -99.1013),
    "Iztapalapa": (19.3575, -99.0924),
    "Cerro de la Estrella": (19.3556, -99.0858),
    "UAM-I": (19.3508, -99.0747),
    "Constitución de 1917": (19.3456, -99.0637),
    "Patriotismo": (19.4059, -99.1802),
    "Chilpancingo": (19.4061, -99.1685),
    "Lázaro Cárdenas": (19.4071, -99.1442),
    "Mixiuhca": (19.4081, -99.1129),
    "Velódromo": (19.4087, -99.1026),
    "Ciudad Deportiva": (19.4082, -99.0913),
    "Puebla": (19.4074, -99.0825),
    "Agrícola Oriental": (19.3998, -99.0709),
    "Canal de San Juan": (19.3888, -99.0558),
    "Tepalcates": (19.3792, -99.0461),
    "Guelatao": (19.3721, -99.0357),
    "Peñón Viejo": (19.3662, -99.0175),
    "Acatitla": (19.3648, -99.0057),
    "Santa Marta": (19.3598, -98.9959),
    "Los Reyes": (19.3591, -98.9768),
    "La Paz": (19.3503, -98.9608),
    "Buenavista": (19.4468, -99.1531),
    "Lagunilla": (19.4435, -99.1311),
    "Tepito": (19.4428, -99.1238),
    "Ricardo Flores Magón": (19.4369, -99.1039),
    "Romero Rubio": (19.4411, -99.0938),
    "Deportivo Oceanía": (19.4508, -99.0789),
    "Bosque de Aragón": (19.4582, -99.0694),
    "Villa de Aragón": (19.4619, -99.0617),
    "Nezahualcóyotl": (19.4721, -99.0543),
    "Impulsora": (19.4856, -99.0487),
    "Río de los Remedios": (19.4907, -99.0461),
    "Múzquiz": (19.5013, -99.0418),
    "Ecatepec": (19.5162, -99.0361),
    "Olímpica": (19.5222, -99.0326),
    "Plaza Aragón": (19.5298, -99.0289),
    "Ciudad Azteca": (19.5348, -99.0274),
    "Insurgentes Sur": (19.3742, -99.1798),
    "Hospital 20 de Noviembre": (19.3728, -99.1699),
    "Parque de los Venados": (19.3678, -99.1538),
    "Eje Central": (19.3608, -99.1466),
    "Mexicaltzingo": (19.3582, -99.1238),
    "Culhuacán": (19.3408, -99.1068),
    "San Andrés Tomatlán": (19.3338, -99.0968),
    "Lomas Estrella": (19.3248, -99.0888),
    "Calle 11": (19.3178, -99.0788),
    "Periférico Oriente": (19.3188, -99.0688),
    "Tezonco": (19.3088, -99.0558),
    "Olivos": (19.3058, -99.0478),
    "Nopalera": (19.3008, -99.0388),
    "Zapotitlán": (19.2958, -99.0288),
    "Tlaltenco": (19.2908, -99.0188),
    "Tláhuac": (19.2868, -99.0138),
}

In [163]:
# Se especializa nuevamente Problem, porqué la h() está hardcode en GraphAStarProblem
class MetroProblem(Problem[str, str]):

  def __init__(
      self,
      initial_state: str,
      goal_state: str,
      graph: dict[str, set[str]],
  ) -> None:
    super().__init__(initial_state, goal_state)
    self.graph: dict[str, set[str]] = graph

  def is_goal(self, state: str) -> bool:
    return self.goal_test(state)

  def get_actions(self, state: str) -> list[str]:
    return list(self.graph.get(state, []))

  def apply_action(self, state: str, action: str) -> str:
    return action

  def h(self, state: str) -> float:
    return haversine_distance(station_coordinates[state], station_coordinates[self.goal_state])

In [164]:
ccp_problem: MetroProblem[str, str] = MetroProblem(
    "Cuatro Caminos", "Pantitlán", metro_graph
)

pt_problem: MetroProblem[str, str] = MetroProblem(
    "Politécnico", "Taxqueña", metro_graph
)

zo_problem: MetroProblem[str, str] = MetroProblem(
    "Zapata", "Oceanía", metro_graph
)

In [165]:
ccp_result, ccp_reached = hill_climbing(ccp_problem)
print("Ruta CCP:", " -> ".join(ccp_result.path()))
print("¿Llegó?:", ccp_reached)

pt_result, pt_reached = hill_climbing(pt_problem)
print("Ruta PT:", " -> ".join(pt_result.path()))
print("¿Llegó?:", pt_reached)

zo_result, zo_reached = hill_climbing(zo_problem)
print("Ruta ZO:", " -> ".join(zo_result.path()))
print("¿Llegó?:", zo_reached)

Ruta CCP: Cuatro Caminos -> Panteones -> Tacuba -> San Joaquín
¿Llegó?: False
Ruta PT: Politécnico -> Instituto del Petróleo -> Vallejo -> Consulado -> Morelos -> Candelaria -> Fray Servando -> Jamaica -> Santa Anita -> Coyuya -> Iztacalco -> Apatlaco -> Aculco -> Escuadrón 201
¿Llegó?: False
Ruta ZO: Zapata -> División del Norte -> Eugenia -> Etiopía -> Centro Médico -> Lázaro Cárdenas -> Chabacano -> Jamaica -> Fray Servando -> Candelaria -> San Lázaro -> Ricardo Flores Magón -> Romero Rubio -> Oceanía
¿Llegó?: True
